知识点	必须掌握的 API	用途
- 数据集划分	train_test_split	划分训练集和测试集
- 缺失值处理	SimpleImputer	填充缺失数据
- 特征标准化	StandardScaler	缩放数值特征
- 类别编码	OneHotEncoder	将类别特征转换为数值
- 文本向量化	CountVectorizer、TfidfVectorizer	将文本转换成数值特征
- 模型评估	accuracy_score、classification_report、confusion_matrix	评价分类效果
- 交叉验证	cross_val_score	评估模型的泛化能力
- 超参数搜索	GridSearchCV、RandomizedSearchCV	搜索更合适的模型参数
- 流水线	Pipeline、ColumnTransformer	串联预处理与模型，减少数据泄漏风险
- 模型保存	joblib	保存和加载训练好的模型
-

In [2]:
import numpy as np
import pandas as pd

In [7]:
# 1、数据集划分	train_test_split	划分训练集和测试集

from sklearn.model_selection import train_test_split

data = np.random.randn(100, 3)
X = data[:, :2]
y = data[:, 2]
print("data.shape: ", data.shape)
print("X.shape: ", X.shape)
print("y.shape: ", y.shape)

# train:validation:test=8:1:1
train_X, tmp_X, train_y, tmp_y = train_test_split(X, y, test_size=0.2, random_state=23)
validation_X, test_X, validation_y, test_y = train_test_split(tmp_X, tmp_y, test_size=0.5, random_state=23)
print("train_X.shape: ", train_X.shape)
print("validation_X.shape: ", validation_X.shape)
print("test_X.shape: ", test_X.shape)

data.shape:  (100, 3)
X.shape:  (100, 2)
y.shape:  (100,)
train_X.shape:  (80, 2)
validation_X.shape:  (10, 2)
test_X.shape:  (10, 2)


In [13]:
# 2、缺失值处理	SimpleImputer	填充缺失数据
# API	作用
# SimpleImputer(strategy="mean")	创建均值填充器
# imputer.fit(X_train)	计算训练集的填充值
# imputer.transform(X_test)	用已有填充值处理新数据
# imputer.fit_transform(X_train)	计算填充值并填充训练集
# imputer.statistics_	查看每列计算出的填充值

from sklearn.impute import SimpleImputer
X = np.array([
    [25, 5000],
    [np.nan, 6000],
    [30, np.nan],
    [35, 7000],
    [np.nan, np.nan]
])
# strategy 策略 使用均值来填充缺失值
# 平均值 "mean", 中位数 "median", 出现次数最多的值 "most_frequent", 固定值 "constant"
imputer = SimpleImputer(strategy="mean")
# Fit to data, then transform it.
transform_X = imputer.fit_transform(X)
print("transform_X: ", transform_X)



# 实际训练中是：先使用 fit 来拟合要填充的值，之后在使用 transform 转化原始数据得到新数据
X_train = np.array([
    [25, 5000],
    [np.nan, 6000],
    [30, np.nan],
    [35, 7000],
    [np.nan, np.nan]
])

X_test = np.array([[np.nan, 5000],[1000, np.nan]])

imputer = SimpleImputer(strategy="median")

imputer.fit(X_train)

print("每列的中位数：", imputer.statistics_)

X_train_filled = imputer.transform(X_train)
X_test_filled = imputer.transform(X_test)
print("X_train_filled: ", X_train_filled)
print("X_test_filled: ", X_test_filled)

transform_X:  [[  25. 5000.]
 [  30. 6000.]
 [  30. 6000.]
 [  35. 7000.]
 [  30. 6000.]]
每列的中位数： [  30. 6000.]
X_train_filled:  [[  25. 5000.]
 [  30. 6000.]
 [  30. 6000.]
 [  35. 7000.]
 [  30. 6000.]]
X_test_filled:  [[  30. 5000.]
 [1000. 6000.]]


In [27]:
# 3、特征标准化	StandardScaler	缩放数值特征
# StandardScaler 会对每一列数据进行处理，使得每列的数据都符合均值为0，方差为1的正太分布
# StandardScaler 只会对飞缺失值进行处理，缺失值还是 nan，因此执行 StandardScaler 之前必须先使用 SimpleImputer 来处理缺失值
from sklearn.preprocessing import StandardScaler

data = np.array([
    [10, 1000],
    [11, 2000],
    [12, 3000],
    [13, np.nan],
    [np.nan, 5000],
    [np.nan, np.nan]
])

# 处理缺失值
imputer = SimpleImputer(strategy="mean")
imputer.fit(data)
print("statistics_: ", imputer.statistics_)
imputer_transform_data = imputer.transform(data)
print("imputer_transform_data: ", imputer_transform_data)

# 特征标准化
scaler = StandardScaler()
scaler.fit(imputer_transform_data)

print("每列均值：", scaler.mean_)
print("每列标准差：", scaler.scale_)

data_scaled = scaler.transform(imputer_transform_data)
print("拟合后新数据：", data_scaled)

statistics_:  [  11.5 2750. ]
imputer_transform_data:  [[  10.  1000. ]
 [  11.  2000. ]
 [  12.  3000. ]
 [  13.  2750. ]
 [  11.5 5000. ]
 [  11.5 2750. ]]
每列均值： [  11.5 2750. ]
每列标准差： [9.12870929e-01 1.20761473e+03]
拟合后新数据： [[-1.64316767 -1.44913767]
 [-0.54772256 -0.621059  ]
 [ 0.54772256  0.20701967]
 [ 1.64316767  0.        ]
 [ 0.          1.86317701]
 [ 0.          0.        ]]


In [41]:
# 4、类别编码	OneHotEncoder	将类别特征转换为数值
# 独热编码（One-Hot Encoding）作用是把类别特征转换成机器学习模型能够处理的数值特征。
from sklearn.preprocessing import OneHotEncoder
# 例如：字符串不能直接作为数值输入许多 sklearn 模型。我们可以使用 OneHotEncoder 将它们转换成多个 0/1 特征。

data = np.array([
    ["Beijing"],
    ["Shanghai"],
    ["Shenzhen"],
    ["Beijing"],
    ["hangzhou"]
])

# sparse_output 表示 是否对矩阵进行压缩， 压缩稀疏行，返回普通 NumPy 数组，而不是稀疏矩阵
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(data)
print("encoder.categories_: ", encoder.categories_)
encoded_data = encoder.transform(data)
print("encoded_data: ", encoded_data)

X_test = np.array([
    ["Shenzhen"]
])

print("X_test: ", encoder.transform(X_test))


encoder.categories_:  [array(['Beijing', 'Shanghai', 'Shenzhen', 'hangzhou'], dtype='<U8')]
encoded_data:  [[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]]
X_test:  [[0. 0. 1. 0.]]


In [46]:
# 5、文本向量化	CountVectorizer、TfidfVectorizer	将文本转换成数值特征
# CountVectorizer：统计词语出现次数
# TfidfVectorizer：根据词语的重要性计算权重
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer

data = np.array([
    "I love machine learning",
    "I love Python",
    "Python is great"
])

cv = CountVectorizer()
cv_data = cv.fit_transform(data)
# ['great' 'is' 'learning' 'love' 'machine' 'python']
print("get_feature_names_out: ", cv.get_feature_names_out())
# toarray()：将稀疏矩阵转换成普通 NumPy 数组，方便观察结果。
# arr 使用 get_feature_names_out 的 one-hot 编码。词袋模型（Bag of Words）的基本思想：统计词语出现情况，不考虑词语的先后顺序。
print("cv_data: ", cv_data.toarray())

tfidf = TfidfVectorizer()
tfidf_data = tfidf.fit_transform(data)
print("tfidf_data: ", tfidf.get_feature_names_out())
# 这里的数值不再是词语的原始出现次数，而是 TF-IDF 权重。默认配置还会对每条文本生成的向量进行 L2 归一化，因此每条非零向量的长度为 1。
print("tfidf_data: ", tfidf_data.toarray())


get_feature_names_out:  ['great' 'is' 'learning' 'love' 'machine' 'python']
cv_data:  [[0 0 1 1 1 0]
 [0 0 0 1 0 1]
 [1 1 0 0 0 1]]
tfidf_data:  ['great' 'is' 'learning' 'love' 'machine' 'python']
tfidf_data:  [[0.         0.         0.62276601 0.4736296  0.62276601 0.        ]
 [0.         0.         0.         0.70710678 0.         0.70710678]
 [0.62276601 0.62276601 0.         0.         0.         0.4736296 ]]


In [54]:
# 6、上述1-5的综合使用示例

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.DataFrame({
    "age": [20, 30, 25, 40, 35, 28, 45, 32],
    "salary": [3000, 8000, 5000, 12000, 9000, 4000, 15000, 7000],
    "city": [
        "Beijing", "Shanghai", "Beijing", "Shenzhen",
        "Shanghai", "Shenzhen", "Beijing", "Shanghai"
    ],
    "texts": [
        "I love machine learning", "I love Python", "Python is great", "free prize call now",
        "free money claim prize", "hello how are you", "are we meeting today", "where are from?"
    ],
    "buy": [0, 1, 0, 1, 1, 0, 1, 1]
})

X = df[["age", "salary", "city", "texts"]]
y = df["buy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# 数值特征的处理流程
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# 类别特征的处理流程
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# 文本特征的处理流程
text_pipeline = Pipeline([
    ("encoder", TfidfVectorizer())
])

# 分别处理数值列和类别列
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, ["age", "salary"]),
    ("categorical", categorical_pipeline, ["city"]),
    # TfidfVectorizer 接收的是一维的字符串文本，因此这里 "texts" 外面不要加[]
    ("text", text_pipeline, "texts")
])

# 预处理 + 模型
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

model.fit(X_train, y_train)

print("预测结果：", model.predict(X_test))
print("准确率：", model.score(X_test, y_test))

预测结果： [1 1]
准确率： 0.5


In [ ]:
# 6、模型评估	accuracy_score、classification_report、confusion_matrix	评价分类效果

In [56]:
# 7、交叉验证	cross_val_score	评估模型的泛化能力
# 假设你使用 train_test_split() 将数据随机划分一次，模型准确率为 90%。
# 这个结果可能受到随机划分的影响。如果换一批训练数据，准确率可能只有 75%。
# 交叉验证的思路是：把训练数据分成多份，轮流用其中一份验证，用其余部分训练。

# 这里使用 heart-disease.csv 来进行演示

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# 1. 数据读取
# age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
data = pd.read_csv("../../../data/ml/heart-disease.csv")
X = data.drop(columns=["target"])
y = data["target"]

# 2. 模型定义
# LogisticRegression、RidgeClassifier、SGDClassifier、Perceptron
model = Pipeline([
    ("preprocessor", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000))
])

# 3. 使用 cross 进行交叉验证
# 执行 5 折交叉验证, 使用准确率来验证
# 常见的 scoring 参数：
# 参数	评估指标	常见场景
# "accuracy"	准确率	一般分类任务
# "precision"	精确率	关注预测为正类的样本有多少是真的正类
# "recall"	召回率	关注实际正类有多少被识别出来
# "f1"	F1 分数	需要综合考虑精确率与召回率
# "roc_auc"	ROC 曲线下面积	评估二分类模型的区分能力
# "neg_mean_squared_error"	负均方误差	回归任务，分数越大（越接近 0）越好
scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")

# 4. 查看结果
print("每一折的准确率：", scores)
print("平均准确率：", scores.mean())
print("准确率标准差：", sco3res.std())


每一折的准确率： [0.88292683 0.85365854 0.86341463 0.82439024 0.80487805]
平均准确率： 0.8458536585365855
准确率标准差： 0.027868988989435507
